# Tomorrow Low valuse

In [22]:
import yfinance as yf
import pandas as pd
import numpy as np

from IPython.display import display




In [43]:
import yfinance as yf

stock = input("Enter stock symbol (Ex. TCS, RELIANCE, INFY): ").upper()

stock += ".NS"

print("Downloading data.....")

try:
    df = yf.download(stock, period="10y")

    if df.empty:
        print("Please enter a valid stock symbol. Data not found!")
    else:
        print("Download successful!\n")

        
        filename = f"data/{stock}.csv"
        df.to_csv(filename)

        pd.read_csv(filename)

        display(df)

except Exception as e:
    print("An error occurred while downloading data.")
    print("Error:", e)

Enter stock symbol (Ex. TCS, RELIANCE, INFY):  acmesolar


[*********************100%***********************]  1 of 1 completed

Download successful!



Price,Close,High,Low,Open,Volume
Ticker,ACMESOLAR.NS,ACMESOLAR.NS,ACMESOLAR.NS,ACMESOLAR.NS,ACMESOLAR.NS
Date,,,,,
2024-11-13,252.685257,275.593137,248.043800,250.539210,22939368
2024-11-14,228.429886,259.422872,227.431722,251.537381,5351111
2024-11-18,251.237915,251.237915,227.830969,235.566741,3943111
2024-11-19,263.665039,276.341735,255.579904,260.570724,9862379
2024-11-21,238.561234,266.559726,237.313528,264.962688,4049771
...,...,...,...,...,...
2026-08-05,382.250000,391.049988,378.000000,384.799988,2758030
2026-08-06,375.450012,380.700012,373.000000,380.700012,1423424


In [44]:
predict_df = df.copy()

In [45]:
train_df = df.copy()

In [46]:
train_df["Prev_Close"] = train_df["Close"].shift(1)
train_df["Prev_High"] = train_df["High"].shift(1)
train_df["Prev_Low"] = train_df["Low"].shift(1)
train_df["Prev_Volume"] = train_df["Volume"].shift(1)

In [47]:
predict_df["Prev_Close"] = predict_df["Close"].shift(1)
predict_df["Prev_High"] = predict_df["High"].shift(1)
predict_df["Prev_Low"] = predict_df["Low"].shift(1)
predict_df["Prev_Volume"] = predict_df["Volume"].shift(1)

In [48]:
train_df["Tomorrow_Low"] = train_df["Low"].shift(-1)

In [49]:
train_df.dropna(inplace=True)


In [50]:
df.tail(1)

Price,Close,High,Low,Open,Volume
Ticker,ACMESOLAR.NS,ACMESOLAR.NS,ACMESOLAR.NS,ACMESOLAR.NS,ACMESOLAR.NS
Date,,,,,
2026-08-11,368.450012,370.700012,362.950012,364.0,667352


In [51]:
features = [
   "Open",
    "High",
    "Low",
    "Close",
    "Volume",
    "Prev_Close",
    "Prev_High",
    "Prev_Low",
    "Prev_Volume"
]

In [52]:
X = train_df[features]

y = train_df["Tomorrow_Low"]

In [53]:
test_days = int(train_df.shape[0] * 0.2)

X_train = X.iloc[:-test_days]
X_test = X.iloc[-test_days:]

y_train = y.iloc[:-test_days]
y_test = y.iloc[-test_days:]

In [54]:
print("Train:", X_train.shape)
print("Test :", X_test.shape)

Train: (346, 9)
Test : (86, 9)


In [55]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_test_scaled = scaler.transform(X_test)

In [56]:
from sklearn.linear_model import LinearRegression
model = LinearRegression()

model.fit(X_train_scaled, y_train)

LinearRegression()

In [57]:
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from sklearn.metrics import r2_score

import numpy as np

prediction = model.predict(X_test_scaled)

print("MAE :", mean_absolute_error(y_test, prediction))
print("RMSE :", np.sqrt(mean_squared_error(y_test, prediction)))
print("R2 :", r2_score(y_test, prediction))

MAE : 4.307006769648063
RMSE : 5.353291642592932
R2 : 0.9778924573838329


In [58]:
latest = predict_df[features].tail(2).head(1)

latest_scaled = scaler.transform(latest)

In [59]:
latest

Price,Open,High,Low,Close,Volume,Prev_Close,Prev_High,Prev_Low,Prev_Volume
Ticker,ACMESOLAR.NS,ACMESOLAR.NS,ACMESOLAR.NS,ACMESOLAR.NS,ACMESOLAR.NS,,,,
Date,,,,,,,,,
2026-08-10,369.0,369.600006,362.75,365.799988,564448,368.600006,380.950012,362.75,1278670.0


In [60]:
tomorrow_low = model.predict(latest_scaled)

print("Tomorrow Low Prediction")

print(tomorrow_low[0])

Tomorrow Low Prediction
361.4472277852752


In [61]:
result = pd.DataFrame({
    "Date": y_test.index,
    "Actual": y_test.values,
    "Predicted": prediction
})

result = result.sort_index(ascending=False)
result

,Date,Actual,Predicted
85,2026-08-10,362.950012,361.447228
84,2026-08-07,362.750000,360.589900
83,2026-08-06,362.750000,370.122904
82,2026-08-05,373.000000,373.891245
81,2026-08-04,378.000000,373.742660
...,...,...,...
4,2026-04-17,297.100006,298.021217
3,2026-04-16,288.049988,283.945721
2,2026-04-15,285.750000,280.880644
1,2026-04-13,284.600006,274.991394


In [62]:
display_result = result.copy()

display_result["Date"] = display_result["Date"] + pd.Timedelta(days=1)

display_result.sort_values("Date", ascending=False)

,Date,Actual,Predicted
85,2026-08-11,362.950012,361.447228
84,2026-08-08,362.750000,360.589900
83,2026-08-07,362.750000,370.122904
82,2026-08-06,373.000000,373.891245
81,2026-08-05,378.000000,373.742660
...,...,...,...
4,2026-04-18,297.100006,298.021217
3,2026-04-17,288.049988,283.945721
2,2026-04-16,285.750000,280.880644
1,2026-04-14,284.600006,274.991394
